# 07 — Pure OpenDSS or CEPT?

**Goal:** run the same small source → line → load Case twice and compare the solver path with the CEPT evidence workflow.

**Prediction:** both paths should agree on the numerical load-flow answer. CEPT should add traceability and a saved verification trail around that answer.

**Same numerical answer, stronger traceability with CEPT.**


In [1]:
#@title 1. Setup — run once
#@title 1. Setup — run once
from hashlib import sha256
from urllib.request import urlopen

_bootstrap_url = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/main/public/notebooks/_lesson.py"
_bootstrap = urlopen(_bootstrap_url, timeout=60).read()
if sha256(_bootstrap).hexdigest() != "aa5805c306987984ba7ee64d57763db1938cb06052cf80e0f8f26fa84efba30d":
    raise ValueError("Lesson helper hash mismatch")
exec(compile(_bootstrap, "cept-lesson", "exec"), globals())

CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version
cept-power-studio 0.2.0.dev0
Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.


In [1]:
#@title 2. Inputs — one declared Case for both paths
#@title 2. Inputs — one declared Case for both paths
CASE_PATH = WORKSPACE / "first_circuit_case.json"
CASE_PAYLOAD = first_circuit_case()
CASE_PATH.write_text(json.dumps(CASE_PAYLOAD, indent=2) + "\n", encoding="utf-8")
table(["declared input", "value", "unit"], [("voltage", 12.47, "kV"), ("line length", 1.0, "km"), ("load", 100.0, "kW"), ("power factor", 0.95, "1")])


declared input     value  unit
----------------  -----  ----
voltage            12.47  kV
line length         1.0   km
load              100.0  kW
power factor       0.95  1


In [1]:
#@title 3. Pure OpenDSS — run the solver directly
#@title 3. Pure OpenDSS — run the solver directly
import opendssdirect as dss

for command in [
    'Clear',
    'New Circuit.first basekv=12.47 pu=1.0 phases=3 bus1=source',
    'New Line.line1 bus1=source.1.2.3 bus2=load.1.2.3 phases=3 length=1 units=km r1=0.2 x1=0.4 r0=0.6 x0=1.2 c1=0 c0=0',
    'New Load.load1 bus1=load.1.2.3 phases=3 conn=wye kv=12.47 kw=100 pf=0.95',
    'Set Voltagebases=[12.47]',
    'CalcVoltageBases',
    'Solve',
]:
    dss.Text.Command(command)
pure_converged = bool(dss.Solution.Converged())
dss.Circuit.SetActiveBus('load')
pure_load_voltage = float(dss.Bus.puVmagAngle()[0])
pure_loss_kw = float(dss.Circuit.Losses()[0]) / 1000
assert pure_converged
assert pure_load_voltage > 0
assert pure_loss_kw >= 0


converged: True
load-bus voltage A: 0.999759 pu
active loss: 0.0143 kW


In [1]:
#@title 4. Pure OpenDSS result — values are immediate, evidence is manual
#@title 4. Pure OpenDSS result — values are immediate, evidence is manual
table(
    ['pure OpenDSS result', 'value', 'unit'],
    [
        ('converged', pure_converged, 'bool'),
        ('load-bus voltage, phase A', pure_load_voltage, 'pu'),
        ('active loss', pure_loss_kw, 'kW'),
        ('case fingerprint', 'not provided by pure OpenDSS', '—'),
        ('manifest / verification receipt', 'not provided by default', '—'),
    ],
)


pure OpenDSS result             value                         unit
----------------------  ----------------------------  ----
converged                True                          bool
load-bus voltage A       0.999759                      pu
active loss             0.0143                        kW
case fingerprint        not provided by pure OpenDSS  —
manifest / receipt      not provided by default      —


In [1]:
# 5. CEPT path — same declared Case, guarded workflow
!cept study run first_circuit_case.json \
    --out runs/07-pure-vs-cept \
    --force \
    --format text
!cept study verify runs/07-pure-vs-cept --format text


CEPT study result: FINISHED
Result             Finished the balanced load flow and saved the evidence.
Study type         Load flow (OpenDSS)
Case fingerprint   aaea341ddd26 (matches the case you ran)
Artifacts          runs/07-pure-vs-cept
Status             PASSED

CEPT study check: PASSED
[PASS] Case identity
[PASS] Solver result
[PASS] Saved evidence
Claim              WORKFLOW_VALIDATED


In [1]:
#@title 6. CEPT result — SLD, plot, and persisted bus table
#@title 6. CEPT result — SLD, plot, and persisted bus table
RUN_DIR = WORKSPACE / "runs" / "07-pure-vs-cept"
display_sld(RUN_DIR)


Bus,Phase A,Phase B,Phase C,Status
load,0.9998 pu @ -0.01°,0.9998 pu @ -120.01°,0.9998 pu @ 119.99°,OK
source,1.0000 pu @ 0.00°,1.0000 pu @ -120.00°,1.0000 pu @ 120.00°,OK


In [1]:
#@title 7. Compare the two paths
#@title 7. Compare the two paths
results = read(RUN_DIR / "results.json")
cept_lf = results["load_flow"]
table(
    ['comparison', 'pure OpenDSS', 'CEPT'],
    [
        ('converged', pure_converged, cept_lf['converged']),
        ('load-bus voltage A (pu)', f'{pure_load_voltage:.6f}', f"{next(r['v_pu'] for r in cept_lf['bus_voltages'] if r['bus'] == 'load' and r['phase'] == 1):.6f}"),
        ('active loss (kW)', f'{pure_loss_kw:.4f}', f"{cept_lf['total_loss_kw']:.4f}"),
        ('case fingerprint', 'manual / absent', results['case_fingerprint']),
        ('saved evidence', 'manual / absent', 'results + manifest + receipt'),
    ],
)
assert abs(pure_load_voltage - next(r['v_pu'] for r in cept_lf['bus_voltages'] if r['bus'] == 'load' and r['phase'] == 1)) < 0.001


comparison                  pure OpenDSS                 CEPT
----------------------  --------------------------  --------------------------
converged                 True                       True
load-bus voltage A (pu)   0.999759                   0.999787
active loss (kW)         0.0143                     0.0143
case fingerprint         manual / absent            aaea341ddd26
saved evidence            manual / absent            results + manifest + receipt


## 8. Interpret

The numerical answers are close because both paths ultimately use the same OpenDSS solver. The operational difference is what happens around that solve.

**Pure OpenDSS** is excellent for exploration: the circuit is explicit, the solve is immediate, and the values are easy to inspect. The operator must still arrange paths, remember the Case, compare the right files, and decide independently what counts as evidence.

**CEPT Studio** keeps the same solver-backed idea but adds a bounded workflow: typed Case, declared run directory, persisted result, canonical SLD, plot, fingerprint, manifest, and independent verification. It does not claim that the Case is a real project or that the solver result is field validation.

**What CEPT adds is repeatability and reviewability, not a different power-flow answer.**